# Can the model learn two familiar images?
This diagnostic uses one resistor photo and one wire photo from the **training** split only. It trains a box detector first, then a keypoint detector on exactly the same images. This separates basic object detection from the additional keypoint task.

**Training and validation intentionally use the SAME two images.** The resulting scores measure memorization, not generalization or project accuracy. The original validation and test sets are not used or changed. Nothing is deployed to the app.

Upload the same **breadboard-pilot-ready.zip** used before. Use a T4 GPU and run each cell in order. Both runs use 120 epochs without early stopping or image augmentation, and an optimizer update every batch. A failed diagnostic does not by itself prove which setting or annotation is wrong.

Data: [BME Electronics, Circuit Detection v5](https://universe.roboflow.com/bme-electronics/circuit-detection-eoirn/dataset/5), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Uses the previously documented box expansions. No LEDs are included.

In [ ]:
%pip -q install ultralytics==8.4.156 pyyaml
import torch, ultralytics
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
print(ultralytics.__version__, torch.cuda.get_device_name(0))

## Upload the prepared ZIP and inspect labels
The code selects two training images and builds separate detection and pose folders. Original files are untouched. Colored boxes should surround the intended objects; dots should coincide with the annotated landmarks. Some wire endpoints are free ends outside the board. If the labels are visibly misplaced, stop and share these preview images before training.

In [ ]:
from google.colab import files
from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import display
import io, zipfile, tempfile, yaml, json, math
uploaded = files.upload()
assert len(uploaded) == 1, 'Choose only breadboard-pilot-ready.zip.'
filename, payload = next(iter(uploaded.items()))
work = Path(tempfile.mkdtemp(prefix='circuit_diagnostic_', dir='/content'))
report_dir = work/'report'
report_dir.mkdir()
selected = []
classes = ['breadboard','resistor','wire']
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    assert 'audit.json' in archive.namelist(), 'Upload the prepared pilot ZIP, not the original export.'
    audit = json.loads(archive.read('audit.json'))
    assert audit['classes'] == classes and audit['kpt_shape'] == [8,3]
    candidates = sorted(n for n in archive.namelist() if n.startswith('images/train/') and n.endswith('.jpg'))
    for prefix in ('20240829_124535', '20240912_150903'):
        match = [n for n in candidates if Path(n).name.startswith(prefix)]
        assert len(match) == 1, f'Missing expected diagnostic image: {prefix}'
        name = match[0]
        assert archive.getinfo(name).file_size < 10_000_000
        label_name = name.replace('images/', 'labels/', 1).rsplit('.',1)[0]+'.txt'
        rows = [list(map(float,l.split())) for l in archive.read(label_name).decode().splitlines() if l.strip()]
        assert all(len(v)==29 and v[0] in (0,1,2) and all(math.isfinite(x) for x in v) for v in rows)
        image_bytes = archive.read(name)
        image = Image.open(io.BytesIO(image_bytes)).convert('RGB')
        w,h = image.size
        draw = ImageDraw.Draw(image)
        colors = ['lime','orange','cyan']
        for v in rows:
            color = colors[int(v[0])]
            x,y,bw,bh = v[1:5]
            draw.rectangle(((x-bw/2)*w,(y-bh/2)*h,(x+bw/2)*w,(y+bh/2)*h),outline=color,width=2)
            for i in range(5,29,3):
                px,py,visible = v[i:i+3]
                if visible:
                    draw.ellipse((px*w-2,py*h-2,px*w+2,py*h+2),fill=color)
        image.save(report_dir/(prefix+'_labels.jpg'))
        print('LABEL PREVIEW:',prefix,'— green: board, orange: resistor, cyan: wire')
        display(image)
        selected.append(Path(name).name)
        for task in ('detect','pose'):
            folder = work/task
            (folder/'images'/'train').mkdir(parents=True,exist_ok=True)
            (folder/'labels'/'train').mkdir(parents=True,exist_ok=True)
            (folder/'images'/'train'/Path(name).name).write_bytes(image_bytes)
            label_rows = [v[:5] if task=='detect' else v for v in rows]
            (folder/'labels'/'train'/(Path(name).stem+'.txt')).write_text('\n'.join(' '.join(format(x,'.12g') for x in v) for v in label_rows)+'\n')
for task in ('detect','pose'):
    config = {'path':str(work/task),'train':'images/train','val':'images/train','names':classes}
    if task=='pose':
        config['kpt_shape'] = [8,3]
    (work/task/'data.yaml').write_text(yaml.safe_dump(config))
summary = {'purpose':'MEMORIZATION DIAGNOSTIC ONLY — train and validation are identical', 'images':selected, 'source_audit':audit, 'ultralytics_version':ultralytics.__version__ if 'ultralytics' in globals() else 'not loaded during preparation', 'runs':{}}
print('Prepared two images per task. Validation intentionally reuses training images.')

## Shared diagnostic settings
The two experiments use fresh pretrained models rather than the failed pilot checkpoint. `nbs=2` with `batch=2` avoids accumulating many batches before an optimizer update on this tiny set. `patience=0` disables early stopping. All geometric/color augmentation and mixed precision are disabled to simplify the check. These are diagnostic settings, not a recommended final training recipe.

In [ ]:
from ultralytics import YOLO
import shutil
settings = dict(epochs=120, patience=0, imgsz=640, batch=2, nbs=2, device=0, workers=0, optimizer='AdamW', lr0=0.001, lrf=1.0, warmup_epochs=0.0, weight_decay=0.0, seed=42, amp=False, mosaic=0.0, mixup=0.0, copy_paste=0.0, degrees=0.0, translate=0.0, scale=0.0, shear=0.0, perspective=0.0, flipud=0.0, fliplr=0.0, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, plots=True)
summary['settings'] = settings
def run_diagnostic(task, checkpoint):
    model = YOLO(checkpoint)
    model.train(data=str(work/task/'data.yaml'), project=str(work/'runs'), name=task, **settings)
    last = Path(model.trainer.last)
    assert last.exists(), 'No final checkpoint was produced.'
    trained = YOLO(str(last))
    metrics = trained.val(data=str(work/task/'data.yaml'), split='val', imgsz=640, batch=2, device=0, workers=0, project=str(work/'evaluations'), name=task, plots=True)
    run_dir = report_dir/task
    run_dir.mkdir(exist_ok=True)
    summary['runs'][task] = {'checkpoint':str(last), 'same_image_metrics':{k:float(v) for k,v in metrics.results_dict.items()}, 'predictions':[]}
    for sample in sorted((work/task/'images'/'train').glob('*.jpg')):
        prediction = trained.predict(str(sample), conf=0.25, verbose=False)[0]
        print(task.upper(),sample.name,'— SAME IMAGE USED FOR TRAINING')
        annotated = Image.fromarray(prediction.plot(kpt_line=False)[..., ::-1])
        annotated.save(run_dir/(sample.stem+'_prediction.jpg'))
        display(annotated)
        summary['runs'][task]['predictions'].append({'image':sample.name,'objects':json.loads(prediction.to_json())})
    for source_dir in (Path(model.trainer.save_dir), Path(metrics.save_dir)):
        for item in source_dir.iterdir():
            if item.is_file() and item.suffix.lower() in ('.csv','.yaml','.png','.jpg'):
                shutil.copy2(item,run_dir/(source_dir.parent.name+'_'+item.name))
    (report_dir/'diagnostic_summary.json').write_text(json.dumps(summary,indent=2))
    print(task,'diagnostic saved. These scores are NOT held-out accuracy.')

## 1. Can it learn component boxes?
Run this cell and wait for all 120 epochs and the prediction images. This simpler baseline ignores keypoints while keeping the same boxes and classes.

In [ ]:
run_diagnostic('detect','yolo11n.pt')

## 2. Can it learn the boxes and keypoints together?
Run after the detection cell finishes. This uses the original eight keypoint slots. Compare the predictions and learning curves with the box-only run; do not treat a score difference alone as proof of the cause.

In [ ]:
run_diagnostic('pose','yolo11n-pose.pt')

## Download the diagnostic report
Send **circuit_diagnostic_report.zip** back in chat. It contains metrics, label previews, predicted images, settings, and training curves. It does not contain deployment weights. This cell can also run after just the detection experiment if you need to save progress.

If boxes are learned but keypoints are not, inspect the pose targets/loss before another full run. If neither is learned, inspect labels, gradients/optimizer steps, and resolution. If both fit, the remaining problem is performance on independent layouts; these two familiar images cannot establish that.

In [ ]:
assert summary['runs'], 'Run at least the detection experiment first.'
(report_dir/'diagnostic_summary.json').write_text(json.dumps(summary,indent=2))
archive_path = shutil.make_archive('/content/circuit_diagnostic_report','zip',report_dir)
files.download(archive_path)
print('Saved circuit_diagnostic_report.zip. Keep the original pilot model/results too.')